# AI-Generated vs Real Image Classifier

**Course:** CPS 4801 — Artificial Intelligence  
**Author:** Dahana Moz Ruiz      
**Institution:** Kean University

## Project Overview
This notebook compares **ResNet-50** and **VGG-16** for classifying images as AI-generated or real photographs using transfer learning. It includes training, evaluation, confusion matrix analysis, Monte Carlo uncertainty estimation, and user-submitted image testing.

**Dataset:** [AI Generated Images vs Real Images — Kaggle](https://www.kaggle.com/datasets/cashbowman/ai-generated-images-vs-real-images)  
- 975 total images: 539 AI-generated, 436 real  
- Labels: 0 = AI-generated, 1 = Real

## Notebook Structure
1. Setup & Configuration
2. Dataset Loading & Custom Dataset Class
3. Data Visualization
4. ResNet-50 — Model Definition & Training
5. VGG-16 — Model Definition & Training
6. Confusion Matrix Evaluation
7. Monte Carlo Uncertainty Estimation
8. User-Submitted Image Testing

## 1. Setup & Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── UPDATE THIS PATH TO YOUR GOOGLE DRIVE LOCATION ──────────────────
DATA_DIR = "/content/drive/MyDrive/AI_vs_Real_Images"  # folder containing AiArtData/ and RealArt/
MODEL_DIR = "/content/drive/MyDrive/AI_vs_Real_Images"  # where .pth files will be saved
# ─────────────────────────────────────────────────────────────────────

import os
print(f"Data directory: {DATA_DIR}")
print(f"Model directory: {MODEL_DIR}")

In [ ]:
!pip install imagehash --quiet

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from io import BytesIO
import imagehash
import cv2
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Dataset Loading & Custom Dataset Class

In [ ]:
ai_dir   = os.path.join(DATA_DIR, "AiArtData/AiArtData")
real_dir = os.path.join(DATA_DIR, "RealArt/RealArt")

ai_images   = [os.path.join(ai_dir,   img) for img in os.listdir(ai_dir)   if img.endswith(('.jpg', '.jpeg', '.png'))]
real_images = [os.path.join(real_dir, img) for img in os.listdir(real_dir) if img.endswith(('.jpg', '.jpeg', '.png'))]

all_images = ai_images + real_images
all_labels = [0] * len(ai_images) + [1] * len(real_images)  # 0 = AI, 1 = Real

print(f"AI-generated images: {len(ai_images)}")
print(f"Real images:         {len(real_images)}")
print(f"Total:               {len(all_images)}")

train_images, test_images, train_labels, test_labels = train_test_split(
    all_images, all_labels, test_size=0.2, random_state=SEED
)
print(f"\nTrain: {len(train_images)} | Test: {len(test_images)}")

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = Image.open(self.images[idx]).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

train_dataset = CustomDataset(train_images, train_labels, transform=transform)
test_dataset  = CustomDataset(test_images,  test_labels,  transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

# Verify
images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}")
print(f"Labels:      {labels[:8].tolist()}")

## 3. Data Visualization

Perceptual hash distribution to visualize structural differences between AI-generated and real images.

In [ ]:
def compute_hashes(image_paths):
    hashes = {}
    for path in image_paths:
        img = cv2.imread(path)
        if img is not None:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            phash = imagehash.phash(Image.fromarray(gray))
            hashes[os.path.basename(path)] = int(str(phash), 16)
    return hashes

print("Computing perceptual hashes...")
real_hashes = compute_hashes(real_images[:100])   # sample for speed
ai_hashes   = compute_hashes(ai_images[:100])

plt.figure(figsize=(10, 5))
plt.hist(list(real_hashes.values()), bins=50, alpha=0.6, label='Real Images', color='steelblue')
plt.hist(list(ai_hashes.values()),   bins=50, alpha=0.6, label='AI-Generated Images', color='coral')
plt.xlabel('Perceptual Hash Value')
plt.ylabel('Frequency')
plt.title('Distribution of Perceptual Hash Values: Real vs AI-Generated')
plt.legend()
plt.tight_layout()
plt.show()

## 4. ResNet-50 — Model Definition & Training

ResNet-50 uses residual (skip) connections to avoid the vanishing gradient problem, enabling effective training of deep networks. Pretrained on ImageNet, fine-tuned for binary classification.

In [ ]:
class ResNet50(nn.Module):
    def __init__(self, num_classes=2):
        super(ResNet50, self).__init__()
        self.resnet = models.resnet50(pretrained=True)
        num_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Linear(num_features, num_classes)

    def forward(self, x):
        return self.resnet(x)

resnet_model = ResNet50(num_classes=2).to(device)
resnet_criterion = nn.CrossEntropyLoss()
resnet_optimizer = optim.Adam(resnet_model.parameters(), lr=0.001)

print("ResNet-50 loaded. Final layer:", resnet_model.resnet.fc)

In [ ]:
def train_model(model, train_loader, criterion, optimizer, num_epochs=10, model_name="model"):
    loss_history, accuracy_history = [], []

    for epoch in range(num_epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total   += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        epoch_loss = running_loss / len(train_loader)
        epoch_acc  = 100 * correct / total
        loss_history.append(epoch_loss)
        accuracy_history.append(epoch_acc)
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%")

    print("Finished Training")
    return loss_history, accuracy_history

resnet_loss, resnet_acc = train_model(
    resnet_model, train_loader, resnet_criterion, resnet_optimizer, num_epochs=10, model_name="ResNet50"
)

resnet_path = os.path.join(MODEL_DIR, 'resnet50_model.pth')
torch.save(resnet_model.state_dict(), resnet_path)
print(f"ResNet-50 saved to {resnet_path}")

In [ ]:
def plot_training(loss_history, accuracy_history, model_name):
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.plot(loss_history, label='Training Loss', color='coral')
    plt.xlabel('Epoch'); plt.ylabel('Loss')
    plt.title(f'{model_name}: Training Loss')

    plt.subplot(1, 2, 2)
    plt.plot(accuracy_history, label='Training Accuracy', color='steelblue')
    plt.xlabel('Epoch'); plt.ylabel('Accuracy (%)')
    plt.title(f'{model_name}: Training Accuracy')

    plt.tight_layout()
    plt.show()

plot_training(resnet_loss, resnet_acc, 'ResNet-50')

## 5. VGG-16 — Model Definition & Training

VGG-16 uses 13 consecutive 3×3 convolutional layers followed by 3 fully connected layers. Feature extraction layers are frozen; only the classifier is fine-tuned.

In [ ]:
class VGG16(nn.Module):
    def __init__(self, num_classes=2):
        super(VGG16, self).__init__()
        self.features  = models.vgg16(pretrained=True).features
        self.avgpool   = nn.AdaptiveAvgPool2d((7, 7))
        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096), nn.ReLU(True), nn.Dropout(),
            nn.Linear(4096, 4096),        nn.ReLU(True), nn.Dropout(),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

vgg_model = VGG16(num_classes=2).to(device)

# Freeze feature extraction layers — only train classifier
for param in vgg_model.features.parameters():
    param.requires_grad = False

vgg_criterion = nn.CrossEntropyLoss()
vgg_optimizer = optim.Adam(vgg_model.parameters(), lr=0.001)

print("VGG-16 loaded. Feature layers frozen.")

In [ ]:
vgg_loss, vgg_acc = train_model(
    vgg_model, train_loader, vgg_criterion, vgg_optimizer, num_epochs=10, model_name="VGG16"
)

vgg_path = os.path.join(MODEL_DIR, 'VGG_16.pth')
torch.save(vgg_model.state_dict(), vgg_path)
print(f"VGG-16 saved to {vgg_path}")

plot_training(vgg_loss, vgg_acc, 'VGG-16')

## 6. Confusion Matrix Evaluation

In [ ]:
def generate_confusion_matrix(model, test_loader, device):
    model.eval()
    all_predictions, all_labels = [], []

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return confusion_matrix(all_labels, all_predictions)

def plot_confusion_matrix(cm, class_names, title):
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, cmap='Blues', fmt='g',
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted labels')
    plt.ylabel('True labels')
    plt.title(title)
    plt.tight_layout()
    plt.show()

def accuracy_from_cm(cm):
    return np.trace(cm) / np.sum(cm) * 100

class_names = ['AI', 'Real']

resnet_cm = generate_confusion_matrix(resnet_model, test_loader, device)
vgg_cm    = generate_confusion_matrix(vgg_model,    test_loader, device)

plot_confusion_matrix(resnet_cm, class_names, 'ResNet-50 Confusion Matrix')
plot_confusion_matrix(vgg_cm,    class_names, 'VGG-16 Confusion Matrix')

print(f"ResNet-50 Test Accuracy: {accuracy_from_cm(resnet_cm):.2f}%")
print(f"VGG-16    Test Accuracy: {accuracy_from_cm(vgg_cm):.2f}%")

## 7. Monte Carlo Uncertainty Estimation

Monte Carlo dropout runs each model in train mode (activating dropout) for 10 iterations per sample, then computes mean prediction, variance, standard deviation, and 95% confidence intervals to quantify prediction uncertainty.

In [ ]:
def monte_carlo_uncertainty(model, dataloader, num_iterations=10, max_batches=1):
    all_results = []

    for batch_idx, (images, labels) in enumerate(dataloader):
        if batch_idx >= max_batches:
            break
        images = images.to('cpu')
        predictions = []

        for _ in range(num_iterations):
            model.train()   # enable dropout
            with torch.no_grad():
                outputs = model(images)
                probs   = torch.nn.functional.softmax(outputs, dim=1)
                predictions.append(probs.cpu().numpy())

        predictions = np.array(predictions)
        mean_pred   = np.mean(predictions, axis=0)
        var_pred    = np.var(predictions,  axis=0)
        all_results.append((mean_pred, var_pred, labels))

    return all_results

def print_uncertainty(results, model_name):
    print(f"\n{'='*60}")
    print(f"{model_name} Model Uncertainty (Monte Carlo, 10 iterations)")
    print(f"{'='*60}")
    for batch_idx, (mean_pred, var_pred, labels) in enumerate(results):
        print(f"\nBatch {batch_idx + 1}:")
        for i in range(min(5, len(labels))):  # show first 5 samples
            std_dev = np.sqrt(var_pred[i])
            ci_low  = mean_pred[i] - 1.96 * std_dev
            ci_high = mean_pred[i] + 1.96 * std_dev
            print(f"  Sample {i+1}:")
            print(f"    True Label:          {labels[i].item()}")
            print(f"    Mean Prediction:     {np.round(mean_pred[i], 4)}")
            print(f"    Variance:            {np.round(var_pred[i], 6)}")
            print(f"    Std Dev:             {np.round(std_dev, 6)}")
            print(f"    95% Conf Interval:   [{np.round(ci_low, 4)}, {np.round(ci_high, 4)}]")

# Load saved models for uncertainty testing
resnet_eval = ResNet50(2)
resnet_eval.load_state_dict(torch.load(resnet_path, map_location='cpu'))

vgg_eval = VGG16(2)
vgg_eval.load_state_dict(torch.load(vgg_path, map_location='cpu'))

resnet_mc = monte_carlo_uncertainty(resnet_eval, test_loader, num_iterations=10, max_batches=1)
vgg_mc    = monte_carlo_uncertainty(vgg_eval,    test_loader, num_iterations=10, max_batches=1)

print_uncertainty(resnet_mc, "ResNet-50")
print_uncertainty(vgg_mc,    "VGG-16")

## 8. User-Submitted Image Testing

Test both models on images from outside the dataset. Accepts local file paths or URLs.

In [ ]:
def load_image(image_path):
    if image_path.startswith('http://') or image_path.startswith('https://'):
        response = requests.get(image_path)
        image = Image.open(BytesIO(response.content)).convert('RGB')
    else:
        image = Image.open(image_path).convert('RGB')
    return image

def preprocess(image):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])
    return transform(image).unsqueeze(0)

def predict(model, image_tensor):
    model.eval()
    with torch.no_grad():
        outputs = model(image_tensor)
        _, predicted = torch.max(outputs, 1)
    return "AI-generated" if predicted.item() == 0 else "Real Art"

def test_image(image_path):
    image = load_image(image_path)
    tensor = preprocess(image)

    resnet_pred = predict(resnet_eval, tensor)
    vgg_pred    = predict(vgg_eval,    tensor)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, model_name, pred in zip(axes, ['ResNet-50', 'VGG-16'], [resnet_pred, vgg_pred]):
        ax.imshow(image)
        ax.set_title(f"{model_name}\nPrediction: {pred}", fontsize=13)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

    print(f"ResNet-50 → {resnet_pred}")
    print(f"VGG-16    → {vgg_pred}")

# Example — replace with your own image path or URL
image_path = input("Enter image path or URL: ")
test_image(image_path)